# CyberGuard: Exploratory Data Analysis

This notebook represents the exploration phase of the CyberGuard project. We investigate the raw authentication logs to find suspicious patterns, failed login rates, and geographic anomalies.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sqlite3

%matplotlib inline
sns.set_theme(style="darkgrid")

## 1. Loading the Raw Data

In [ ]:
df = pd.read_csv("../data/raw/auth_logs.csv")
df.head()

## 2. Basic Dataset Statistics

In [ ]:
print(f"Total logs: {len(df)}")
print(f"Unique users: {df['username'].nunique()}")
print(f"Success/Failure distribution:\n{df['status'].value_counts(normalize=True)}")

## 3. Threat Hunting & Risk Rules Setup

Let's identify anomalies such as:
- Brute force attempts (excessive failed attempts from the same IP)
- Impossible Travel (logins from multiple distinct countries within the dataset)

In [ ]:
# Users with high failed login attempts
failed_attempts = df[df['status'] == 'Failed'].groupby('username').size().sort_values(ascending=False)
print("Top users by failed logins:")
print(failed_attempts.head())

In [ ]:
# IPs running brute-force attacks
failed_ips = df[df['status'] == 'Failed'].groupby('ip_address').size().sort_values(ascending=False)
print("Top IPs by failed logins:")
print(failed_ips.head())

In [ ]:
# Users with logins from multiple countries
user_countries = df.groupby('username')['country'].nunique().sort_values(ascending=False)
print("Users logging in from multiple countries:")
print(user_countries[user_countries > 1])

## 4. Querying the Processed Database

Once the production script runs, it stores data in SQLite. Let's inspect the database.

In [ ]:
conn = sqlite3.connect("../data/cyberguard.db")
db_profiles = pd.read_sql_query("SELECT * FROM user_risk_profiles ORDER BY max_risk_score DESC LIMIT 10", conn)
conn.close()

db_profiles